In [0]:
#Read Silver Tables
df_posts = spark.read.table("log_analytics.silver.posts_logs")
df_comments = spark.read.table("log_analytics.silver.comments_logs")

In [0]:
#Import Functions
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
#Total Logs & Errors
df_error_stats = df_posts.agg(
    count("*").alias("total_logs"),
    sum("is_error").alias("total_errors"),
)

In [0]:
df_error_stats = df_error_stats.withColumn(
    "error_rate",
    col("total_errors") / col("total_logs")
)

In [0]:
#Add timestamp
df_error_stats = df_error_stats.withColumn("processed_time", current_timestamp())

In [0]:
#Write Error Statistics
df_error_stats.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true")\
    .saveAsTable("log_analytics.gold.error_statistics")

In [0]:
#LOG LENGTH DISTRIBUTION
#Length Statistics
df_length_stats = df_posts.agg(
    avg("body_length").alias("avg_length"),
    stddev("body_length").alias("stddev_length"),
    variance("body_length").alias("variance_length")
)

In [0]:
df_length_stats = df_length_stats.withColumn("processed_time", current_timestamp())

In [0]:
#write
df_length_stats.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("log_analytics.gold.length_statistics")

In [0]:
#USER ACTIVITY
#Comments per User
df_user_activity = df_comments.groupBy("email").agg(
    count("*").alias("total_comments")
)

In [0]:
#Add Timestamp
df_user_activity = df_user_activity.withColumn("processed_time", current_timestamp())

In [0]:
if spark.catalog.tableExists("log_analytics.gold.user_activity"):

    gold_user = DeltaTable.forName(spark, "log_analytics.gold.user_activity")

    gold_user.alias("target").merge(
        df_user_activity.alias("source"),
        "target.email = source.email"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

else:

    df_user_activity.write.format("delta") \
        .mode("overwrite") \
        .option("mergeSchema", "true")\
        .saveAsTable("log_analytics.gold.user_activity")

In [0]:

#USER STATISTICS
#Activity statistics
df_user_stats = df_user_activity.agg(
    avg("total_comments").alias("avg_comments"),
    stddev("total_comments").alias("stddev_comments"),
    variance("total_comments").alias("variance_comments")
)

In [0]:
df_user_stats = df_user_stats.withColumn("processed_time", current_timestamp())

In [0]:
df_user_stats.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true")\
    .saveAsTable("log_analytics.gold.user_statistics")

In [0]:
#ERROR BY USER
#Error per User
df_error_by_user = df_posts.groupBy("userId").agg(
    sum("is_error").alias("total_user_errors"),
    count("*").alias("total_user_posts")
)

In [0]:
df_error_by_user = df_error_by_user.withColumn("processed_time", current_timestamp())

In [0]:
#Write Error by User
if spark.catalog.tableExists("log_analytics.gold.error_by_user"):

    gold_error = DeltaTable.forName(spark, "log_analytics.gold.error_by_user")

    gold_error.alias("target").merge(
        df_error_by_user.alias("source"),
        "target.userId = source.userId"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

else:

    df_error_by_user.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable("log_analytics.gold.error_by_user")

In [0]:
#Join Data
df_joined = df_posts.join(
    df_comments,
    df_posts.id == df_comments.postId,
    "inner"
)

In [0]:
#Select Columns
df_final = df_joined.select(
    df_posts.id.alias("post_id"),
    df_posts.userId,
    df_posts.title,                      
    df_posts.body_length,               
    df_posts.log_level,
    df_posts.is_error,
    df_comments.id.alias("comment_id"), 
    df_comments.email,
    df_comments.name.alias("commenter_name"),
    df_comments.activity_type,
    df_posts.processed_time
)

In [0]:
df_final.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true")\
    .saveAsTable("log_analytics.gold.logs_joined")

In [0]:
# Posts per userId — feeds bar chart
df_posts_per_user = df_posts.groupBy("userId").agg(
    count("*").alias("total_posts")
).orderBy("userId")

In [0]:
df_posts_per_user = df_posts_per_user.withColumn("processed_time", current_timestamp())

In [0]:
df_posts_per_user.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("log_analytics.gold.posts_per_user")

In [0]:
# Count comments per post, join with post title
df_comment_counts = df_comments.groupBy("postId").agg(
    count("*").alias("total_comments")
)

In [0]:
# Join with posts to get titles
df_top_posts = df_posts.select(
    col("id").alias("postId"),
    col("title"),
    col("userId")
).join(df_comment_counts, on="postId", how="inner") \
 .orderBy(desc("total_comments"))

In [0]:
df_top_posts = df_top_posts.withColumn("processed_time", current_timestamp())

In [0]:
df_top_posts.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("log_analytics.gold.top_posts_by_comments")


In [0]:
# Body length per post — each row is one post with its length
# Power BI uses this to draw a bar per postId showing how long each post body is
df_post_length = df_posts.select(
    col("id").alias("postId"),
    col("userId"),
    col("title"),
    col("body_length"),
    col("log_level")
).orderBy("postId")

In [0]:
df_post_length = df_post_length.withColumn("processed_time", current_timestamp())

In [0]:
df_post_length.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("log_analytics.gold.post_length_distribution")

In [0]:
# Error details — one row per comment with email and issue classification
# issue_type is derived: if email is invalid → Invalid Email
#                        if body is empty → Empty Comment
#                        if post is error → Error Log
#                        else → Valid
df_error_details = df_final.select(
    col("email"),
    when(col("email").rlike(r"^[\w.+-]+@[\w-]+\.[\w.]+$"), "Valid Email")
    .otherwise("Invalid Email").alias("email_valid"),
    col("is_error"),
    col("log_level")
).withColumn(
    "issue_type",
    when(col("email_valid") == "Invalid Email", "Invalid Email")
    .when(col("is_error") == 1, "Error Log")
    .otherwise("Valid")
)

In [0]:
df_error_details = df_error_details.withColumn("processed_time", current_timestamp())

In [0]:
df_error_details.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("log_analytics.gold.error_details")

In [0]:
# userId × postId comment count — Matrix heatmap in Power BI
# Each row = one combination of userId and postId with comment count
df_engagement = df_final.groupBy("userId", "post_id").agg(
    count("comment_id").alias("comment_count"),
    sum("is_error").alias("error_count")
).orderBy("userId", "post_id")


In [0]:
df_engagement = df_engagement.withColumn("processed_time", current_timestamp())

In [0]:
df_engagement.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("log_analytics.gold.user_engagement")

In [0]:
# Top 5 users by total posts + total comments combined
# Join posts count and comments count per userId
df_user_posts_count = df_posts.groupBy("userId").agg(
    count("*").alias("user_posts")
)

In [0]:
df_user_comments_count = df_final.groupBy("userId").agg(
    count("comment_id").alias("user_comments"),
    sum("is_error").alias("user_errors")
)

In [0]:
df_top5 = df_user_posts_count.join(df_user_comments_count, on="userId", how="inner") \
    .withColumn("total_activity", col("user_posts") + col("user_comments")) \
    .orderBy(desc("total_activity")) \
    .limit(5)

In [0]:
df_top5 = df_top5.withColumn("processed_time", current_timestamp())


In [0]:
df_top5.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("log_analytics.gold.top5_users_contribution")